#Power Electric Consumption LSTM

At this code, We'll do a LSTM modeling of a power consumption historic and evaluate the model loss quality.

##1) Importing Frameworks and Libraries

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader

##2) Importing the dataset

In [ ]:
df = pd.read_csv('/content/household_power_consumption.csv', sep=';',
                 parse_dates={'Datetime': ['Date', 'Time']},
                 infer_datetime_format=True)

/tmp/ipython-input-2366230927.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv('/content/household_power_consumption.csv', sep=';',
/tmp/ipython-input-2366230927.py:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv('/content/household_power_consumption.csv', sep=';',
/tmp/ipython-input-2366230927.py:1: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/household_power_consumption.csv', sep=';',
/tmp/ipython-input-2366230927.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified

##3) ETL

Let's check the missing values

In [ ]:
print(df[df.isna().any(axis=1)])

                   Datetime Global_active_power Global_reactive_power Voltage  \
6839    2006-12-21 11:23:00                   ?                     ?       ?   
6840    2006-12-21 11:24:00                   ?                     ?       ?   
19724   2006-12-30 10:08:00                   ?                     ?       ?   
19725   2006-12-30 10:09:00                   ?                     ?       ?   
41832   2007-01-14 18:36:00                   ?                     ?       ?   
...                     ...                 ...                   ...     ...   
1990185 2010-09-28 19:09:00                   ?                     ?       ?   
1990186 2010-09-28 19:10:00                   ?                     ?       ?   
1990187 2010-09-28 19:11:00                   ?                     ?       ?   
1990188 2010-09-28 19:12:00                   ?                     ?       ?   
2027411 2010-10-24 15:35:00                   ?                     ?       ?   

        Global_intensity Su

Let's standarize the missing values

In [ ]:
df.replace('?', np.nan, inplace=True)

Here, We'll format the collumns to float

In [ ]:
for col in df.columns:
    if col not in ['Datetime', 'Sub_metering_3']:
        df[col] = df[col].astype(float)

Then change the index

In [ ]:
df.set_index('Datetime', inplace=True)

Now, We'll query the dataframe information

In [ ]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2075259 entries, 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Data columns (total 7 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Global_active_power    2049280 non-null  float64
 1   Global_reactive_power  2049280 non-null  float64
 2   Voltage                2049280 non-null  float64
 3   Global_intensity       2049280 non-null  float64
 4   Sub_metering_1         2049280 non-null  float64
 5   Sub_metering_2         2049280 non-null  float64
 6   Sub_metering_3         2049280 non-null  float64
dtypes: float64(7)
memory usage: 126.7 MB


Here, hour and weekday collumns are created and will substitute missing values

In [ ]:
#Carregar dados (ajuste o path conforme necessário)
#Criar variáveis de tempo
df['hour'] = df.index.hour
df['weekday'] = df.index.weekday  # Segunda=0, Domingo=6

#Calcular médias por hora e dia da semana
group_means = df.groupby(['hour', 'weekday']).mean(numeric_only=True)

#Função para imputar
def fill_missing(row):
    if row.isnull().any():
        grp = (row.name.hour, row.name.weekday())
        if grp in group_means.index:
            return row.fillna(group_means.loc[grp])
        else:
            return row.fillna(df.mean(numeric_only=True))
    return row

#Aplicar imputação somente nas colunas numéricas (excluindo hour e weekday)
cols = df.select_dtypes(include='number').columns.difference(['hour','weekday'])
df[cols] = df[cols].apply(fill_missing, axis=1)


We need to drop hour and weekday collumns after used

In [ ]:
df = df.drop(columns=['hour', 'weekday'])

##4) Modeling and Training

In [ ]:
# Configurações básicas
SEED = 42
WINDOW_SIZE = 24 * 7  # Uma semana de dados
BATCH_SIZE = 32
EPOCHS = 20

# Selecionar apenas o consumo global
consumption = df['Global_active_power'].values.reshape(-1, 1)

# Normalização dos dados
scaler = MinMaxScaler()
consumption_scaled = scaler.fit_transform(consumption)

class PowerConsumptionDataset(Dataset):
    def __init__(self, data, window_size):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size]
        y = self.data[idx + self.window_size]
        return x, y

# Criar conjuntos de treino e teste
train_size = int(len(consumption_scaled) * 0.8)
train_dataset = PowerConsumptionDataset(consumption_scaled[:train_size], WINDOW_SIZE)
test_dataset = PowerConsumptionDataset(consumption_scaled[train_size:], WINDOW_SIZE)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Definir o modelo LSTM
class PowerLSTM(nn.Module):
    def __init__(self):
        super(PowerLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=50, num_layers=2, batch_first=True)
        self.fc = nn.Linear(50, 1)

    def forward(self, x):
        # Garantir que o estado inicial esteja no mesmo dispositivo que x
        h0 = torch.zeros(2, x.size(0), 50, device=x.device).detach()
        c0 = torch.zeros(2, x.size(0), 50, device=x.device).detach()
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

# Configurar dispositivo (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PowerLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Treinamento
def train(model, device, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        data = data.view(-1, WINDOW_SIZE, 1)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

# Avaliação
def evaluate(model, device, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            data = data.view(-1, WINDOW_SIZE, 1)
            output = model(data)
            total_loss += criterion(output, target).item()
    return total_loss / len(loader)

# Loop de treinamento
best_loss = float('inf')
for epoch in range(EPOCHS):
    train_loss = train(model, device, train_loader, optimizer, criterion)
    test_loss = evaluate(model, device, test_loader, criterion)

    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), 'power_consumption_model.pth')

    print(f'Epoch {epoch+1}, Train Loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}')

Epoch 1, Train Loss: 0.000627, Test Loss: 0.000390
Epoch 2, Train Loss: 0.000575, Test Loss: 0.000449
Epoch 3, Train Loss: 0.000544, Test Loss: 0.000345
Epoch 4, Train Loss: 0.000525, Test Loss: 0.000344
Epoch 5, Train Loss: 0.000512, Test Loss: 0.000332
Epoch 6, Train Loss: 0.000501, Test Loss: 0.000323
Epoch 7, Train Loss: 0.000493, Test Loss: 0.000322
Epoch 8, Train Loss: 0.000486, Test Loss: 0.000326
Epoch 9, Train Loss: 0.000481, Test Loss: 0.000317
Epoch 10, Train Loss: 0.000477, Test Loss: 0.000321
Epoch 11, Train Loss: 0.000474, Test Loss: 0.000317
Epoch 12, Train Loss: 0.000470, Test Loss: 0.000313
Epoch 13, Train Loss: 0.000468, Test Loss: 0.000312


KeyboardInterrupt: 